# ML model preparation (YOLO)

<div style="text-align: justify">

Notebook used to train and validate an object detection model, especially a YOLOv5 model. Model is trained on the exported LARD datasets from [`data-export.ipynb`](./data-export.ipynb) notebook.
</div>

> Notebook inspired from G. Delhomme's work. [[Github]](https://github.com/geoffrey-g-delhomme/lard-yolov8)

## General helpers

In [1]:
from pathlib import Path
from typing import (
    Union,
    Tuple,
    List
)

import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import cv2
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from ultralytics import YOLO

from swmf.data import create_yolo_datayaml

<div style='text-align: justify', class="alert alert-danger">

Make sure to have the correct path to exported LARD dataset, from the previous [`data-export.ipynb`](./data-export.ipynb) notebook.
</div>

In [2]:
PATH_TO_EXPORTED_LARD = "../data/datasets/lard_512x512_ICPR2026"

In [3]:
### Reproductibility ###
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
########################

In [4]:
### Training params  ###
D_IMGSZ = [512, 512]  # [W, H]
D_BATCH = 64
D_N_EPOCHS = 20
D_MODEL_NAME = "yolov5n.pt"  # Use smallest versions for RT (yolov5n, yolov8n, ...)
D_MODEL_TASK = "detect"
########################

In [5]:
def YOLO_train(
        dataset_path: Path,
        model_name: Union[str, Path] = D_MODEL_NAME,
        imgsz: Union[int, Tuple[int,int]] = D_IMGSZ,
        batch: int = -1,
        epochs: int = D_N_EPOCHS,
        resume: bool = False,
        **kwargs,
): 
    """
    Train a YOLO model on given dataset.

    Args:
        dataset_path (Path): the path to the train dataset
        model_name (Union[str, Path]): the name/path of/to the model
        imgsz (Union[int, Tuple[int, int]]): the size of the images
        batch (int): the batch size
        epochs (int): the number of training epochs
        resume (bool): whether to resume from past checkpoint or start from scratch

    Returns:
        (ultralytics.DetMetrics) The training result.
    """
    data_path = dataset_path / 'data.yaml'
    devices = ','.join([str(i) for i in range(torch.cuda.device_count())])
    batches = torch.cuda.device_count() * batch

    result = YOLO(model_name).train(
        data=data_path,
        imgsz=imgsz,
        batch=batches,
        device=devices,
        epochs=epochs,
        resume=resume,
        **kwargs,
    )
    return result


def YOLO_valid(
        dataset_path: Path,
        model: YOLO,
): 
    """
    Compute metrics for trained model on the validation dataset.
    
    Args:
        dataset_path (Path): The path to the dataset
        model (YOLO): The trained YOLO model

    Returns:
        The metrics for validation.
    """
    result = model.val(data=dataset_path / 'data.yaml')
    return result


def YOLO_test(
        dataset_path: Path,
        model: YOLO,
):
    """
    Compute metrics for trained model on the test dataset.

    Args:
        dataset_path (Path): The path to the dataset
        model (YOLO): The trained YOLO model

    Returns:
        The metrics for test.
    """
    result = model.val(data=dataset_path / 'data_test.yaml')
    return result


def YOLO_predict(
        sources_path: Union[Path, List[Path]],
        model: YOLO,
        **kwargs,
):
    """
    Launch YOLO prediction on given data.

    Args:
        sources_path (Union[Path, List[Path]]): The path to the input data.
        model (YOLO): The trained YOLO model.

    Returns:
        The results of the YOLO prediction routine.
    """
    devices = ','.join([str(i) for i in range(torch.cuda.device_count())])
    result = model.predict(
        source=sources_path,
        device=devices,
        save=True,
        save_txt=True,
        show_labels=True,
        verbose=False,
        **kwargs,
    )
    return result


def YOLO_export(
        model: YOLO,
        export_format: str = "onnx",
):
    """
    Export a YOLO model to any supported format
    """
    model.export(format=export_format, dynamic=True, simplify=True)


In [6]:
def _util_cp_training_result_data(src: Path, dst: Path):
    """
    Copy content of 'src' folder into 'dst'. 

    Args:
        src (Path): The path to Ultralytics 'save_dir'
        dst (Path): The path to your model 'save_dir'

    Note:
        The function saves the model weights, the training args and the generated plots.
    """
    if not src.exists():
        raise ValueError(f"Source path does not exist. {src.as_posix()}")

    dst.mkdir(parents=True, exist_ok=True)
    sub = dst / "training_info"
    sub.mkdir(parents=True, exist_ok=True)

    for f in src.iterdir():
        if f.is_file() and f.suffix in ['.yaml', '.png']:
            shutil.copy2(f, sub)
            print(f"✅ Copied {f.name} to {sub.as_posix()}")
    
    for f in (src / "weights").iterdir():
        if f.is_file() and f.suffix in [".pt", ".onnx"]:
            shutil.copy2(f, dst)
            print(f"✅ Copied {f.name} to {dst.as_posix()}")


def _util_rm_yolo_training_folder(root: Path = Path("runs")):
    """
    Remove the YOLO 'runs/' directory.

    Args:
        root (Path): The path to base runs/ folder.
    """
    if not root.exists():
        print(f"Nothing to delete. {root} does not exist.")
        return
    
    shutil.rmtree(root)
    print("✅ Deleted YOLO 'runs/' directory.")
    

<div style="text-align: justify">

Launch YOLO training below.
</div>

In [7]:
lard_data_dpath = Path(PATH_TO_EXPORTED_LARD).resolve()
SPLITS = [
    "split_trainval", 
    "split_trainval_per_runway", 
    "split_trainval_per_airport"
]

### EDIT HERE ###
SPLIT_INDX = 1  # BY DEFAULT WE KEEP THE SPLIT BY RUNWAYS
#################

split_fpath = lard_data_dpath / f"{SPLITS[SPLIT_INDX]}.csv"
split_dname = SPLITS[SPLIT_INDX]

## YOLO training

In [ ]:
# Define path to yolo training data (data.yaml for split!)
yolo_data_dpath = create_yolo_datayaml(
    dataset_dpath=lard_data_dpath,
    yolo_task=D_MODEL_TASK,
    split_fpath=split_fpath,
    split_dname=split_dname,
)
yolo_data_dpath

In [ ]:
# Define path to yolo training save directory
yolo_save_dpath = Path("../data/models").resolve() / D_MODEL_TASK / D_MODEL_NAME.rpartition('.')[0] / (lard_data_dpath.stem + "_" + split_fpath.stem) / f"{D_N_EPOCHS:03d}_epochs"
yolo_save_dpath.as_posix()

In [21]:
### EDIT HERE ###
TRAIN_STATE = 0  # {0: train from scratch, 1: train from chckpnt, 2: no train}
#################

In [ ]:
# Launch YOLO training according to command state
match TRAIN_STATE:
    case 0:
        result = YOLO_train(
            dataset_path=yolo_data_dpath,
            model_name=D_MODEL_NAME,
            imgsz=D_IMGSZ,
            batch=-1,
            epochs=D_N_EPOCHS,
            # patience=3,
            # resume=False,
        )
    case 1:
        result = YOLO_train(
            dataset_path=yolo_data_dpath,
            model_name=yolo_save_dpath / "last.pt",
            # imgsz=D_IMGSZ,
            # batch=D_BATCH,
            epochs=D_N_EPOCHS,
            resume=True,
        )
    case _:
        print("No training required.")

# Delete YOLO runs/ folder if necessary
if TRAIN_STATE <= 1:
    _util_cp_training_result_data(result.save_dir, yolo_save_dpath)
    _util_rm_yolo_training_folder(Path("../runs/").resolve())

### Evaluate the model

In [ ]:
model = YOLO(yolo_save_dpath / "best.pt")
model.fuse()
model.info(verbose=True)

In [ ]:
metrics_valid = YOLO_valid(yolo_data_dpath, model)

print("=== [Valid] metrics ===")
print(metrics_valid)

In [ ]:
metrics_tests = YOLO_test(yolo_data_dpath, model)

print("=== [Tests] metrics ===")
print(metrics_tests)

Save the evaluation results into `./results` directory

In [26]:
def _util_cp_eval_result_data(src: Path, dst: Path):
    """
    """
    assert src.exists(), f"Src path does not exist... {src.as_posix()}"

    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if (src/f).is_file():
            shutil.copy2(src/f, dst)

In [ ]:
path_to_results = Path('../results').resolve()
path_to_results = path_to_results / PATH_TO_EXPORTED_LARD.split('/')[-1] / D_MODEL_TASK / D_MODEL_NAME.rpartition('.')[0] / (lard_data_dpath.stem + "_" + split_fpath.stem) / f"{D_N_EPOCHS:03d}_epochs"
path_to_results.as_posix()

In [28]:
# Copy valid results
valid_dst_dpath = path_to_results / split_dname / 'valid'
valid_src_dpath = Path(metrics_valid.save_dir).resolve()

_util_cp_eval_result_data(valid_src_dpath, valid_dst_dpath)

In [29]:
# Copy tests results
tests_dst_dpath = path_to_results / split_dname / 'test'
tests_src_dpath = Path(metrics_tests.save_dir).resolve()

_util_cp_eval_result_data(tests_src_dpath, tests_dst_dpath)

In [ ]:
# Clean ./runs yolo directory
_util_rm_yolo_training_folder(Path("../runs/").resolve())

Export the YOLO model to ONNX

In [ ]:
YOLO_export(model)

## YOLO prediction

In [32]:
def draw_bbox(lab_filepath: Path, img_filepath: Path = None, ax = None, is_quiet=False, *plt_args, **plt_kwargs):
    """
    """
    sample = pd.read_csv(lab_filepath.as_posix(), delimiter=' ', header=None)
    bbox = sample.iloc[0].to_numpy()[1:]

    if img_filepath is None:
        img_filepath = lab_filepath.parent.parent.parent / "images" / lab_filepath.parent.stem / f"{lab_filepath.stem}.jpg"
    
    image = np.array(cv2.cvtColor(cv2.imread(img_filepath.as_posix()), cv2.COLOR_BGR2RGB))
    h,w,d = image.shape
    bbox[0] *= w
    bbox[1] *= h
    bbox[2] *= w
    bbox[3] *= h

    bbox_xyxy = [
        bbox[0] - bbox[2]/2., 
        bbox[1] - bbox[3]/2., 
        bbox[0] + bbox[2]/2., 
        bbox[1] + bbox[3]/2., 
    ]
    image = cv2.rectangle(image, 
                          (int(bbox_xyxy[0]), int(bbox_xyxy[1])),
                          (int(bbox_xyxy[2]), int(bbox_xyxy[3])),
                          color=(255, 0, 0), thickness=2)
    
    if not is_quiet:
        if ax is None:
            fig, ax = plt.subplots(1, 1, *plt_args, **plt_kwargs)
            ax.imshow(image)
            ax.axis('off')
            fig.tight_layout()
        else:
            ax.imshow(image)
            ax.axis('off')
    return image

In [33]:
### Reproducibility ###
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
#######################

In [ ]:
path_to_test_images = lard_data_dpath / "images" / "test"
path_to_test_images

In [ ]:
n_samples = 10

test_imgs_relpath = np.random.choice([f for f in os.listdir(path_to_test_images)], size=n_samples)
test_imgs_abspath = [path_to_test_images / f for f in test_imgs_relpath]
results = YOLO_predict(test_imgs_abspath, model, imgsz=D_IMGSZ, max_det=1)

In [36]:
# Copy tests results
predict_dst_dpath = path_to_results / split_dname / 'predict'
predict_src_dpath = Path(results[0].save_dir).resolve()

_util_cp_eval_result_data(
    predict_src_dpath, 
    predict_dst_dpath
)

_util_cp_eval_result_data(
    predict_src_dpath / 'labels', 
    predict_dst_dpath / 'labels'
)

In [ ]:
# Delete /runs directory
_util_rm_yolo_training_folder(Path("../runs/").resolve())

## Exploration

*(independent section)*

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [9]:
import torch

In [ ]:
# Define path to yolo training save directory
yolo_path = Path("../data/models/detect")
yolo_path.as_posix()

In [ ]:
yolov5_detector = YOLO(yolo_path / "yolov5n_LARD_20epochs.pt")
yolov5_detector.fuse()
yolov5_detector.info(verbose=True)
yolov5_detector.eval();

In [ ]:
yolov5_detector.model.model[-1]

In [ ]:
test_imgs_path = Path('../data/datasets/lard_512x512_ICPR2026/images/test')
test_imgs = [
    test_imgs_path / "000001.jpg",
    test_imgs_path / "000002.jpg",
    test_imgs_path / "000428.jpg",
]

def logits_hook(module, i_data, o_data):
    global logits
    logits = o_data

hook = yolov5_detector.model.model[-1].register_forward_hook(logits_hook)

with torch.no_grad():
    result = yolov5_detector.predict(test_imgs, imgsz=512)

hook.remove()

In [ ]:
print(len(result))
print(result[0].boxes.data[0,:4])
print(result[1].boxes.data[0,:4])

In [ ]:
print(len(logits))
print(logits[0].shape)
print(logits[1][0].shape)
print(logits[1][1].shape)
print(logits[1][2].shape)
# print(logits[1][2][0])

In [ ]:
torch.cat([l.view(*l.shape[:2], -1) for l in logits[1]], dim=2).shape

In [ ]:
ll = logits[0].detach().cpu().numpy()

sorted_idx = np.argsort(-ll[:,4,:], axis=1)
sorted_log = np.zeros_like(ll)
for b in range(ll.shape[0]):
    sorted_log[b,:,:] = ll[b,:, sorted_idx[b,:]].T
sorted_log[:,:,0]

In [21]:
# 1. Extract raw bboxes (logits[0])
raw_bboxes = logits[0][:,:4,:].permute(0, 2, 1).detach().cpu().numpy()
raw_logits = torch.cat([l.view(*l.shape[:2], -1) for l in logits[1]], dim=2).detach().cpu().numpy()

# 2. Find matching bbox in final preds
match_indice = []
for b in range(len(result)):
    final_bboxes = result[b].boxes.data[:,:4].detach().cpu().numpy()
    batch_indice = []

    for final_bbox in final_bboxes:
        d = np.sqrt(np.sum((raw_bboxes[b] - final_bbox)**2, axis=1))
        batch_indice.append(np.argmin(d))
    
    match_indice.append(np.array(batch_indice))

# 3. Extract correspondig logits
final_logits = []
for b in range(len(result)):
    batch_logits = raw_logits[b, :, match_indice[b]]  # 65, n
    final_logits.append(batch_logits)

final_logits = np.concatenate(final_logits, 0)

In [ ]:
final_logits